<center>
<img src="https://supportvectors.ai/logo-poster-transparent.png" width=400px style="opacity:0.8">
</center>


In [1]:
%run supportvectors-common.ipynb


<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors. All rights reserved. <blockquote>This notebook is the intellectual property of SupportVectors, and part of its training material. 
Only the participants in SupportVectors workshops are allowed to study the notebooks for educational purposes currently, but is prohibited from copying or using it for any other purposes without written permission.

<b> These notebooks are chapters and sections from Asif Qamar's textbook that he is writing on Data Science. So we request you to not circulate the material to others.</b>
 </blockquote>
 <hr/>
</div>



# Lab 02b — Retrieval as Scoring: The Memory Stream

## Learning goals

1. Treat retrieval as a **scoring problem**, not a nearest-neighbour lookup.
2. Implement the canonical memory-stream score from *Generative Agents*: **recency + importance + relevance**.
3. Show that **all three terms are load-bearing** — ablate each and watch a distinct failure appear.
4. Show why **normalization** matters: without it, a term's *scale* (not its *signal*) decides the ranking.

## Theory you need

Lab 02 decided *what gets written*. This lab decides *what gets read back*. The naive answer — cosine similarity, top-k — has a failure mode you have already met in production RAG: **recent chatter crowds out old, load-bearing facts.** "Ada loved the ramen last week" will always out-similarity "Ada has a severe peanut allergy" on a dinner query — and only one of those facts can hurt her.

The *Generative Agents* paper (Park et al., 2023) buried the fix inside its **memory stream**:

```
score(fact, query) = w_r · recency(fact) + w_i · importance(fact) + w_s · similarity(fact, query)
```

| Term | What it measures | Failure when it is missing |
|------|------------------|----------------------------|
| **Recency** | exponential decay since last access (0.995 per hour) | the agent dwells on stale context |
| **Importance** | how consequential the fact is, judged at *write* time | allergies lose to ramen |
| **Similarity** | embedding relevance to the query | recent trivia surfaces everywhere |

Two details carry the design:

1. **Each term is min–max normalized to [0, 1] across the candidate set** before weighting. Raw cosine lives in a narrow band while raw recency spans the whole interval; unnormalized, one term dominates by scale alone.
2. **Importance is assigned at write time** — Lab 02's salience gate already produced it. Retrieval *spends* what the write-policy *saved*. A store with garbage importances cannot be rescued by any weights.

> **Note:** this lab makes **no LLM calls** — only the sentence-transformer you already used in Lab 02. Everything here is retrieval arithmetic, so it runs fast even when the class endpoint is busy.


In [2]:
# Supporting Python lives in src/memory (installed via `uv sync`).
# Notebooks only contain the lab narrative and exercises.

from datetime import UTC, datetime, timedelta

from memory import FactStore, load_lab_env

load_lab_env()
store = FactStore()


def add_aged(text: str, *, importance: float, days_old: float) -> None:
    """Add a fact, then back-date it (teaching hack — real facts age on their own)."""
    fact = store.add(text, importance=importance, provenance="seed-corpus")
    stamp = (datetime.now(UTC) - timedelta(days=days_old)).isoformat()
    fact.created_at = stamp
    fact.updated_at = stamp


# (text, importance, days_old) — a plausible store after months of Lab-02-style writes.
CORPUS = [
    ("Ada has a severe peanut allergy.",                                    0.98, 120),
    ("Ada is vegetarian.",                                                  0.90, 200),
    ("Ada lives in Austin, Texas.",                                         0.80, 300),
    ("Ada works as a data engineer at a logistics startup.",                0.70, 250),
    ("Ada prefers window seats on long flights.",                           0.70,  60),
    ("Ada prefers email over phone calls.",                                 0.60, 150),
    ("Ada's weekly team sync is on Friday at 3pm.",                         0.60,  30),
    ("Ada is training for a 10k race in November.",                         0.50,  40),
    ("Ada's sister is visiting in August.",                                 0.40,  20),
    ("Ada is thinking about trying the Italian bistro that just opened downtown.", 0.30, 1),
    ("Ada bought a new espresso machine and loves it.",                     0.30,  10),
    ("Ada loved the ramen at Kinme Noodle Bar last week.",                  0.25,   5),
    ("Ada had dinner with Priya at the new sushi place on 5th Street.",     0.20,   3),
    ("Ada said the tacos from the food truck near work were great.",        0.20,   2),
    ("Ada mentioned Sunday brunch with her book club was fun.",             0.15,   6),
]

for text, importance, days_old in CORPUS:
    add_aged(text, importance=importance, days_old=days_old)

print(f"Seeded {len(store)} facts.")
print("Oldest:", store.all()[0].text)
print("Newest:", store.all()[-1].text)


Seeded 15 facts.
Oldest: Ada lives in Austin, Texas.
Newest: Ada is thinking about trying the Italian bistro that just opened downtown.


## Part A — Baseline: cosine similarity, top-k

Query the store the way Lab 02's reconcile step did — pure semantic similarity. The question every dinner recommender must answer correctly is buried way below the top 5: watch where the allergy actually lands.


In [3]:
QUERY = "Suggest a restaurant for Ada's dinner tonight."

baseline = store.search_scored(QUERY, k=len(store))

print(f"Query: {QUERY}\n")
print(f"{'#':>2}  {'cosine':>6}  fact")
for rank, (sim, fact) in enumerate(baseline, 1):
    print(f"{rank:>2}  {sim:6.3f}  {fact.text}")

sim_rank = next(
    rank for rank, (_, f) in enumerate(baseline, 1) if "peanut" in f.text.lower()
)
print(f"\nPeanut allergy ranks #{sim_rank} of {len(baseline)} by cosine alone.")
if sim_rank <= 3:
    print("(Your embedding model rates it unusually high — lucky. The structural problem")
    print(" remains: nothing in this score knows the allergy is load-bearing.)")
else:
    print("Top of the list: recent food chatter. The one fact that can cause an ER visit")
    print("is below the fold — similarity measures topicality, not consequence.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Query: Suggest a restaurant for Ada's dinner tonight.

 #  cosine  fact
 1   0.653  Ada is thinking about trying the Italian bistro that just opened downtown.
 2   0.597  Ada had dinner with Priya at the new sushi place on 5th Street.
 3   0.536  Ada is vegetarian.
 4   0.532  Ada lives in Austin, Texas.
 5   0.522  Ada mentioned Sunday brunch with her book club was fun.
 6   0.514  Ada's sister is visiting in August.
 7   0.511  Ada said the tacos from the food truck near work were great.
 8   0.511  Ada loved the ramen at Kinme Noodle Bar last week.
 9   0.484  Ada prefers window seats on long flights.
10   0.424  Ada prefers email over phone calls.
11   0.415  Ada has a severe peanut allergy.
12   0.414  Ada is training for a 10k race in November.
13   0.397  Ada's weekly team sync is on Friday at 3pm.
14   0.396  Ada bought a new espresso machine and loves it.
15   0.341  Ada works as a data engineer at a logistics startup.

Peanut allergy ranks #11 of 15 by cosine alone.
Top of th

## Part B — The memory-stream score

Three components per fact, each **min–max normalized across the candidate set**, then weighted:

- **similarity** — the cosine score you just saw;
- **recency** — `0.995 ** hours_since_last_touch` (the Generative Agents decay);
- **importance** — stored on the `Fact` at write time by Lab 02's salience gate.

We start with the paper's choice: **equal weights**. Expect an *improvement*, not a miracle — equal weights still let the four recent dinner anecdotes gang up. Tuning is Part D's job.


In [4]:
DECAY_PER_HOUR = 0.995  # Generative Agents' decay factor


def hours_since(iso_timestamp: str) -> float:
    then = datetime.fromisoformat(iso_timestamp)
    return max(0.0, (datetime.now(UTC) - then).total_seconds() / 3600.0)


def minmax(values: list[float]) -> list[float]:
    """Scale to [0, 1] across the candidate set. No spread -> the term carries no signal."""
    lo, hi = min(values), max(values)
    if hi - lo < 1e-12:
        return [0.5] * len(values)
    return [(v - lo) / (hi - lo) for v in values]


def memory_stream_rank(store, query: str, *, weights: dict, k: int = 5, normalize: bool = True):
    """Generative-Agents retrieval: weighted recency + importance + similarity."""
    scored = store.search_scored(query, k=len(store))  # every active fact + cosine
    if not scored:
        return []
    sims = [sim for sim, _ in scored]
    recs = [DECAY_PER_HOUR ** hours_since(fact.updated_at) for _, fact in scored]
    imps = [fact.importance for _, fact in scored]
    if normalize:
        sims, recs, imps = minmax(sims), minmax(recs), minmax(imps)
    rows = []
    for i, (_, fact) in enumerate(scored):
        total = (
            weights["similarity"] * sims[i]
            + weights["recency"] * recs[i]
            + weights["importance"] * imps[i]
        )
        rows.append({"fact": fact, "total": total, "sim": sims[i], "rec": recs[i], "imp": imps[i]})
    rows.sort(key=lambda r: r["total"], reverse=True)
    return rows[:k]


def show_ranking(rows, title: str) -> None:
    print(title)
    print(f"{'#':>2}  {'total':>6}  {'sim':>5}  {'rec':>5}  {'imp':>5}  fact")
    for rank, r in enumerate(rows, 1):
        print(f"{rank:>2}  {r['total']:6.3f}  {r['sim']:5.2f}  {r['rec']:5.2f}  {r['imp']:5.2f}  {r['fact'].text}")
    print()


def rank_of(rows, needle: str):
    for rank, r in enumerate(rows, 1):
        if needle.lower() in r["fact"].text.lower():
            return rank
    return None


# --- run it ---
EQUAL = {"similarity": 1.0, "recency": 1.0, "importance": 1.0}
rows = memory_stream_rank(store, QUERY, weights=EQUAL, k=len(store))
show_ranking(rows[:8], "Memory-stream score, equal weights (top 8):")

full_rank = rank_of(rows, "peanut")
print(f"Allergy rank: cosine-only #{sim_rank}  ->  memory stream #{full_rank}")
assert full_rank <= sim_rank, (
    "The three-term score should never rank the max-importance fact WORSE than "
    "cosine alone — inspect the tables above; your corpus may need stronger distractors."
)
print("✓ Importance pulled the allergy up the board — but check whether it made top-3 yet.")


Memory-stream score, equal weights (top 8):
 #   total    sim    rec    imp  fact
 1   2.181   1.00   1.00   0.18  Ada is thinking about trying the Italian bistro that just opened downtown.
 2   1.667   0.82   0.79   0.06  Ada had dinner with Priya at the new sushi place on 5th Street.
 3   1.528   0.62   0.00   0.90  Ada is vegetarian.
 4   1.492   0.54   0.89   0.06  Ada said the tacos from the food truck near work were great.
 5   1.395   0.61   0.00   0.78  Ada lives in Austin, Texas.
 6   1.283   0.54   0.62   0.12  Ada loved the ramen at Kinme Noodle Bar last week.
 7   1.239   0.24   0.00   1.00  Ada has a severe peanut allergy.
 8   1.129   0.58   0.55   0.00  Ada mentioned Sunday brunch with her book club was fun.

Allergy rank: cosine-only #11  ->  memory stream #7
✓ Importance pulled the allergy up the board — but check whether it made top-3 yet.


## Part C — Ablations: every term is load-bearing

Kill one term at a time and name the failure that returns. Then re-run equal weights **without normalization** and watch scale — not signal — pick the winners.

| Config | Predicted failure |
|--------|-------------------|
| similarity only | Part A again: consequence is invisible |
| no importance | recent + topical wins; allergy sinks |
| no recency | the store's old core dominates; "what's new" never surfaces |
| no similarity | the same VIP facts answer *every* query, relevant or not |


In [5]:
ABLATIONS = {
    "similarity only":            {"similarity": 1.0, "recency": 0.0, "importance": 0.0},
    "no importance (sim+rec)":    {"similarity": 1.0, "recency": 1.0, "importance": 0.0},
    "no recency (sim+imp)":       {"similarity": 1.0, "recency": 0.0, "importance": 1.0},
    "no similarity (rec+imp)":    {"similarity": 0.0, "recency": 1.0, "importance": 1.0},
}

for name, weights in ABLATIONS.items():
    rows = memory_stream_rank(store, QUERY, weights=weights, k=len(store))
    allergy = rank_of(rows, "peanut")
    show_ranking(rows[:3], f"[{name}]  (allergy at #{allergy})")

# Normalization ablation: same equal weights, raw component scales.
raw = memory_stream_rank(store, QUERY, weights=EQUAL, k=len(store), normalize=False)
show_ranking(raw[:5], "[equal weights, NO normalization]")
norm = memory_stream_rank(store, QUERY, weights=EQUAL, k=len(store))
print("Compare with the normalized top-5 from Part B: raw cosine spans a narrow band while")
print("raw recency spans nearly [0, 1], so freshness buys more score than it earned.")
print("Normalization is what makes the weights MEAN something.")

# Try a different query: with similarity zeroed, notice the top-3 barely changes.
rows = memory_stream_rank(store, "What is Ada's job?", weights=ABLATIONS["no similarity (rec+imp)"], k=3)
show_ranking(rows, "[no similarity] on an unrelated query — same VIPs, wrong answer:")


[similarity only]  (allergy at #11)
 #   total    sim    rec    imp  fact
 1   1.000   1.00   1.00   0.18  Ada is thinking about trying the Italian bistro that just opened downtown.
 2   0.821   0.82   0.79   0.06  Ada had dinner with Priya at the new sushi place on 5th Street.
 3   0.625   0.62   0.00   0.90  Ada is vegetarian.

[no importance (sim+rec)]  (allergy at #13)
 #   total    sim    rec    imp  fact
 1   2.000   1.00   1.00   0.18  Ada is thinking about trying the Italian bistro that just opened downtown.
 2   1.607   0.82   0.79   0.06  Ada had dinner with Priya at the new sushi place on 5th Street.
 3   1.431   0.54   0.89   0.06  Ada said the tacos from the food truck near work were great.

[no recency (sim+imp)]  (allergy at #3)
 #   total    sim    rec    imp  fact
 1   1.528   0.62   0.00   0.90  Ada is vegetarian.
 2   1.395   0.61   0.00   0.78  Ada lives in Austin, Texas.
 3   1.239   0.24   0.00   1.00  Ada has a severe peanut allergy.

[no similarity (rec+imp)]  (

## Part D — Exercise: tune the weights

Your acceptance test, in the spirit of Lab 03's preservation gate: **for a dinner query, both dietary facts must beat the food chatter.**

1. Start from the weights below and confirm the asserts pass.
2. Now find the *boundary*: lower `importance` until the allergy drops out of the top-3. How much headroom did you have?
3. Raise `recency` until chatter re-invades. Which weight is your system most sensitive to?
4. Discussion: these weights are a **product decision** (a medical agent is not a news agent). Where would you put them for *your* use-case — and which query set would you gate releases on?

**Stretch (homework):** implement *reflection* — when the store exceeds N facts, prompt the LLM (via `complete()`) to synthesize 2–3 higher-level insights from clusters of related facts and `add()` them with elevated importance and provenance pointing at the source fact ids. That is the second half of the Generative Agents memory design.


In [16]:
# ── TUNE ME ─────────────────────────────────────────────────────────────
WEIGHTS = {"similarity": 1.0, "recency": 0.5, "importance": 2.0}
# ────────────────────────────────────────────────────────────────────────

rows = memory_stream_rank(store, QUERY, weights=WEIGHTS, k=len(store))
show_ranking(rows[:5], f"Tuned weights {WEIGHTS} (top 5):")

allergy_rank = rank_of(rows, "peanut")
veg_rank = rank_of(rows, "vegetarian")
brunch_rank = rank_of(rows, "brunch")
print(f"allergy #{allergy_rank}   vegetarian #{veg_rank}   brunch chatter #{brunch_rank}")

assert allergy_rank <= 3, "Gate failed: the allergy must be top-3 for a dinner query."
assert brunch_rank > 3, "Gate failed: low-importance chatter should not crowd the top-3."
print("\n✓ Retrieval gate passed: what can hurt Ada now outranks what merely happened lately.")
print("  (Note how far vegetarian climbed too — same mechanism, no special-casing.)")


Tuned weights {'similarity': 1.0, 'recency': 0.5, 'importance': 2.0} (top 5):
 #   total    sim    rec    imp  fact
 1   2.432   0.62   0.00   0.90  Ada is vegetarian.
 2   2.239   0.24   0.00   1.00  Ada has a severe peanut allergy.
 3   2.178   0.61   0.00   0.78  Ada lives in Austin, Texas.
 4   1.861   1.00   1.00   0.18  Ada is thinking about trying the Italian bistro that just opened downtown.
 5   1.785   0.46   0.00   0.66  Ada prefers window seats on long flights.

allergy #2   vegetarian #1   brunch chatter #14

✓ Retrieval gate passed: what can hurt Ada now outranks what merely happened lately.
  (Note how far vegetarian climbed too — same mechanism, no special-casing.)


## Next lab

**Lab 03 — Compaction** turns from the durable store to the *working* tier: what happens when the live conversation itself no longer fits the window — and how a preservation gate keeps lossy compression honest.
